# For Determining Redshift

(WIP)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table


def plot_zphot_zspec(zout_path, zmax=6, outlier_thresh=0.15):
    """
    Create a z_phot vs z_spec diagnostic plot similar to threedhst.eazyPy.zPhot_zSpec,
    including σ_NMAD and outlier fraction.

    Parameters
    ----------
    zout_path : str
        Path to the EAZY output file (photz.zout).
    save_as : str, optional
        Output filename for the plot (default: 'zphot_zspec.png').
    zmax : float, optional
        Maximum redshift for axes (default: 6).
    outlier_thresh : float, optional
        |Δz|/(1+z_spec) threshold for outliers (default: 0.15).
    """
    # --- Load EAZY output
    zout = Table.read(zout_path, format='ascii')

    # --- Identify columns automatically
    colnames = [c.lower() for c in zout.colnames]
    if 'z_spec' in colnames:
        zspec_col = zout.colnames[colnames.index('z_spec')]
    elif 'zspec' in colnames:
        zspec_col = zout.colnames[colnames.index('zspec')]
    else:
        raise ValueError("No spectroscopic redshift column found in .zout file.")

    if 'z_a' in colnames:
        zphot_col = zout.colnames[colnames.index('z_a')]
    elif 'z_phot' in colnames:
        zphot_col = zout.colnames[colnames.index('z_phot')]
    else:
        raise ValueError("No photometric redshift column (z_a / z_phot) found in .zout file.")

    # --- Extract arrays and clean NaNs
    zspec = np.array(zout[zspec_col])
    zphot = np.array(zout[zphot_col])
    mask1 = zspec >= 0
    mask2 = zphot >= 0

    mask = mask1 & mask2
    zspec, zphot = zspec[mask], zphot[mask]

    # --- Compute scatter metrics
    dz = (zphot - zspec) / (1 + zspec)
    dz = dz[np.isfinite(dz)]
    sigma_nmad = 1.48 * np.median(np.abs(dz - np.median(dz)))
    outlier_frac = np.mean(np.abs(dz) > outlier_thresh)

    # --- Plot
    plt.figure(figsize=(6, 6))
    plt.scatter(zspec, zphot, s=16, alpha=0.6, color='royalblue', edgecolor='none', label='Data')
    plt.plot([0, zmax], [0, zmax], 'k--', lw=1, label='1:1')
    plt.xlabel(r'Spectroscopic redshift $z_{\rm spec}$')
    plt.ylabel(r'Photometric redshift $z_{\rm phot}$')
    plt.xlim(0, zmax)
    plt.ylim(0, zmax)
    plt.grid(alpha=0.3)

    # --- Annotate performance
    txt = (rf"$\sigma_{{\rm NMAD}} = {sigma_nmad:.3f}$" + "\n" +
           rf"Outlier frac = {outlier_frac*100:.1f}%")
    plt.text(0.05*zmax, 0.9*zmax, txt, fontsize=11, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    plt.title('EAZY Photometric vs. Spectroscopic Redshift')
    plt.tight_layout()
    # plt.savefig(save_as, dpi=150)
    # plt.close()

    # print(f"✅ Plot saved to {save_as}")
    print(f"σ_NMAD = {sigma_nmad:.4f}, Outlier fraction = {outlier_frac*100:.2f}%")

plot_zphot_zspec('../inputs/OUTPUT/photz.zout', zmax=20)

#Delete the comment on line 1 and delete last line (incomplete)

grizli version: 1.13.2
msaexp version: 0.9.12
